This cell loads the already-preprocessed train/test files from the midterm — train_processed.csv and train_labels.csv (SMOTE-balanced, scaled) and test_processed.csv and test_labels.csv (untouched, imbalanced). It then splits the original 30% test set in half to create a 15% validation set and a 15% test set, both keeping the real-world class imbalance since only training data should be balanced with SMOTE.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

sns.set_style('whitegrid')

# Load the already-processed files from the midterm
X_train = pd.read_csv('train_processed.csv')       # SMOTE-balanced, scaled
y_train = pd.read_csv('train_labels.csv')['HEATWAVE']
X_test_full = pd.read_csv('test_processed.csv')    # untouched, imbalanced, scaled
y_test_full = pd.read_csv('test_labels.csv')['HEATWAVE']

print("Training set (70%):", X_train.shape)
print("Held-out set before val/test split:", X_test_full.shape)

# Split the held-out 30% in half -> 15% validation, 15% test
# Both stay realistic/imbalanced since we're NOT applying SMOTE here
X_val, X_test, y_val, y_test = train_test_split(
    X_test_full, y_test_full, test_size=0.5, random_state=42, stratify=y_test_full
)

print("\nFinal split:")
print("Train:", X_train.shape, "| class balance:", y_train.value_counts().to_dict())
print("Validation:", X_val.shape, "| class balance:", y_val.value_counts().to_dict())
print("Test:", X_test.shape, "| class balance:", y_test.value_counts().to_dict())

Training set (70%): (29206, 29)
Held-out set before val/test split: (6588, 29)

Final split:
Train: (29206, 29) | class balance: {0: 14603, 1: 14603}
Validation: (3294, 29) | class balance: {0: 3129, 1: 165}
Test: (3294, 29) | class balance: {0: 3130, 1: 164}


This cell trains six required classification models Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, KNN, and SVC on the balanced training set from the midterm. Each trained model is stored in a dictionary so it can be reused for evaluation and the ensemble steps later.

AI Disclaimer: AI (Gemini) was used to help write this code cell.
Prompt: "Write code that defines a dictionary of six scikit-learn classification models (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, KNN, SVC), loops through and fits each one on X_train and y_train, and stores the trained models in a dictionary."

In [3]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVC': SVC(probability=True, random_state=42)
}

trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model

print("All models trained.")

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...
Training KNN...
Training SVC...
All models trained.


This cell defines a function that evaluates a model using Accuracy, Precision, Recall, F1-Score, and ROC-AUC, then runs it for every trained model on both the validation and test sets. The results are collected into two comparison tables so all six models can be compared side by side.

AI Disclaimer: AI (Gemini) was used to help write this code cell.
Prompt: "Write a function that takes a trained classification model and a feature/target set, returns a dictionary of Accuracy, Precision, Recall, F1-Score, and ROC-AUC, then loop through a dictionary of trained models to build separate validation and test results tables using this function."

In [4]:
def evaluate_model(model, X, y):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]
    return {
        'Accuracy': accuracy_score(y, preds),
        'Precision': precision_score(y, preds),
        'Recall': recall_score(y, preds),
        'F1-Score': f1_score(y, preds),
        'ROC-AUC': roc_auc_score(y, probs)
    }

val_results = {}
test_results = {}

for name, model in trained_models.items():
    val_results[name] = evaluate_model(model, X_val, y_val)
    test_results[name] = evaluate_model(model, X_test, y_test)

val_df_results = pd.DataFrame(val_results).T
test_df_results = pd.DataFrame(test_results).T

print("Validation Results:")
print(val_df_results)
print("\nTest Results:")
print(test_df_results)

Validation Results:
                     Accuracy  Precision    Recall  F1-Score   ROC-AUC
Logistic Regression  0.993018   0.877660  1.000000  0.934844  0.999750
Decision Tree        1.000000   1.000000  1.000000  1.000000  1.000000
Random Forest        1.000000   1.000000  1.000000  1.000000  1.000000
Gradient Boosting    1.000000   1.000000  1.000000  1.000000  1.000000
KNN                  0.959016   0.554745  0.921212  0.692483  0.974996
SVC                  0.989071   0.834197  0.975758  0.899441  0.999165

Test Results:
                     Accuracy  Precision    Recall  F1-Score   ROC-AUC
Logistic Regression  0.989375   0.827411  0.993902  0.903047  0.999636
Decision Tree        1.000000   1.000000  1.000000  1.000000  1.000000
Random Forest        1.000000   1.000000  1.000000  1.000000  1.000000
Gradient Boosting    1.000000   1.000000  1.000000  1.000000  1.000000
KNN                  0.952338   0.512027  0.908537  0.654945  0.957925
SVC                  0.986642   0.797030  

This cell picks the top 3 models based on validation F1-score, then combines them into a soft Voting Classifier that averages their predicted probabilities. The ensemble is trained on the same balanced training set and evaluated on both validation and test sets using the same metrics function.

AI Disclaimer: AI (Gemini) was used to help write this code cell.
Prompt: "Write code that selects the top 3 models from a validation results table based on F1-Score, combines them into a scikit-learn VotingClassifier with soft voting, fits it on the training data, and evaluates it on validation and test sets."

In [5]:
top_3_names = val_df_results.sort_values('F1-Score', ascending=False).head(3).index.tolist()
print("Top 3 models:", top_3_names)

top_3_estimators = [(name, trained_models[name]) for name in top_3_names]

voting_clf = VotingClassifier(estimators=top_3_estimators, voting='soft')
voting_clf.fit(X_train, y_train)

voting_val_metrics = evaluate_model(voting_clf, X_val, y_val)
voting_test_metrics = evaluate_model(voting_clf, X_test, y_test)

print("Voting Ensemble - Validation:", voting_val_metrics)
print("Voting Ensemble - Test:", voting_test_metrics)

Top 3 models: ['Decision Tree', 'Random Forest', 'Gradient Boosting']
Voting Ensemble - Validation: {'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1-Score': 1.0, 'ROC-AUC': np.float64(1.0)}
Voting Ensemble - Test: {'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1-Score': 1.0, 'ROC-AUC': np.float64(1.0)}


This cell builds a weighted ensemble by scaling each of the top 3 models' predicted probabilities by its validation F1-score, instead of averaging them equally like the Voting Classifier does. It then applies a 0.5 threshold to get final predictions and evaluates the weighted ensemble on both validation and test sets.

AI Disclaimer: AI (Gemini) was used to help write this code cell.
Prompt: "Write code that computes normalized weights for the top 3 models based on their validation F1-scores, uses those weights to combine each model's predicted probabilities into a single weighted probability array, converts that to binary predictions using a 0.5 threshold, and evaluates accuracy, precision, recall, F1, and ROC-AUC on validation and test sets."

In [6]:
# Weight each top model's predicted probabilities by its validation F1-score,
# instead of averaging them equally like the Voting Classifier does.
val_f1_scores = np.array([val_df_results.loc[name, 'F1-Score'] for name in top_3_names])
weights = val_f1_scores / val_f1_scores.sum()
print("Model weights based on validation F1:", dict(zip(top_3_names, weights)))

def bayesian_ensemble_predict(X):
    weighted_probs = np.zeros(len(X))
    for name, weight in zip(top_3_names, weights):
        probs = trained_models[name].predict_proba(X)[:, 1]
        weighted_probs += weight * probs
    return weighted_probs

val_probs = bayesian_ensemble_predict(X_val)
test_probs = bayesian_ensemble_predict(X_test)
val_preds = (val_probs >= 0.5).astype(int)
test_preds = (test_probs >= 0.5).astype(int)

bayesian_val_metrics = {
    'Accuracy': accuracy_score(y_val, val_preds),
    'Precision': precision_score(y_val, val_preds),
    'Recall': recall_score(y_val, val_preds),
    'F1-Score': f1_score(y_val, val_preds),
    'ROC-AUC': roc_auc_score(y_val, val_probs)
}
bayesian_test_metrics = {
    'Accuracy': accuracy_score(y_test, test_preds),
    'Precision': precision_score(y_test, test_preds),
    'Recall': recall_score(y_test, test_preds),
    'F1-Score': f1_score(y_test, test_preds),
    'ROC-AUC': roc_auc_score(y_test, test_probs)
}

print("Bayesian Ensemble - Validation:", bayesian_val_metrics)
print("Bayesian Ensemble - Test:", bayesian_test_metrics)

Model weights based on validation F1: {'Decision Tree': np.float64(0.3333333333333333), 'Random Forest': np.float64(0.3333333333333333), 'Gradient Boosting': np.float64(0.3333333333333333)}
Bayesian Ensemble - Validation: {'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1-Score': 1.0, 'ROC-AUC': np.float64(1.0)}
Bayesian Ensemble - Test: {'Accuracy': 1.0, 'Precision': 1.0, 'Recall': 1.0, 'F1-Score': 1.0, 'ROC-AUC': np.float64(1.0)}


This cell combines the individual model results with both ensemble results into two final comparison tables one for validation, one for test and saves them as CSV files. This is the complete side-by-side comparison of all six models plus the Voting and Bayesian ensembles required for the deliverables.

In [7]:
all_val_results = val_results.copy()
all_val_results['Voting Ensemble'] = voting_val_metrics
all_val_results['Bayesian Ensemble'] = bayesian_val_metrics

all_test_results = test_results.copy()
all_test_results['Voting Ensemble'] = voting_test_metrics
all_test_results['Bayesian Ensemble'] = bayesian_test_metrics

final_val_table = pd.DataFrame(all_val_results).T
final_test_table = pd.DataFrame(all_test_results).T

print("FINAL VALIDATION COMPARISON:")
print(final_val_table)
print("\nFINAL TEST COMPARISON:")
print(final_test_table)

final_val_table.to_csv('validation_comparison_with_leakage.csv')
final_test_table.to_csv('test_comparison_with_leakage.csv')

FINAL VALIDATION COMPARISON:
                     Accuracy  Precision    Recall  F1-Score   ROC-AUC
Logistic Regression  0.993018   0.877660  1.000000  0.934844  0.999750
Decision Tree        1.000000   1.000000  1.000000  1.000000  1.000000
Random Forest        1.000000   1.000000  1.000000  1.000000  1.000000
Gradient Boosting    1.000000   1.000000  1.000000  1.000000  1.000000
KNN                  0.959016   0.554745  0.921212  0.692483  0.974996
SVC                  0.989071   0.834197  0.975758  0.899441  0.999165
Voting Ensemble      1.000000   1.000000  1.000000  1.000000  1.000000
Bayesian Ensemble    1.000000   1.000000  1.000000  1.000000  1.000000

FINAL TEST COMPARISON:
                     Accuracy  Precision    Recall  F1-Score   ROC-AUC
Logistic Regression  0.989375   0.827411  0.993902  0.903047  0.999636
Decision Tree        1.000000   1.000000  1.000000  1.000000  1.000000
Random Forest        1.000000   1.000000  1.000000  1.000000  1.000000
Gradient Boosting    1.0

This cell checks Random Forest's feature importances to investigate why several models scored a perfect 100% on validation and test. It reveals that TMAX, TMIN, and TEMP2M together account for most of the model's decisions, suggesting the HEATWAVE label is directly derived from temperature rather than something the model is genuinely predicting.

In [8]:
# Check which features are driving the "perfect" predictions
rf_model = trained_models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(10))

TMAX      0.353671
TMIN      0.260901
TEMP2M    0.178543
MSLP      0.050963
BLH       0.045829
SOILT1    0.037114
MONTH     0.031277
DEW2M     0.011235
EVAP      0.010316
SOILM1    0.004597
dtype: float64


Finding: TMAX, TMIN, and TEMP2M together account for roughly 79% of Random Forest's decision-making (0.354 + 0.261 + 0.179), far more than any other feature. This confirms the perfect scores above are a form of data leakage. HEATWAVE is just another word for temperature, so tree-based models can nearly reconstruct HEATWAVE's own definition rather than learning genuine predictive patterns from indirect weather signals. The ablation below removes these three columns to get a much better accuracy of the model's performance.

This cell drops TMAX, TMIN, and TEMP2M. The temperature features driving the perfect scores and retrains all six models to see how well they perform using only indirect weather signals instead of features that essentially restate the label's definition. It builds separate validation and test comparison tables for this leak-free version of the problem.

AI Disclaimer: AI (Gemini) was used to help write this code cell.
Prompt: "Write code that drops a list of specified columns from training, validation, and test feature sets, retrains a dictionary of six scikit-learn classification models on the reduced training set, evaluates each on validation and test using an existing evaluation function, and builds separate validation and test comparison tables from the results."

In [9]:
# Ablation: drop the temperature features that directly define HEATWAVE,
# to see how well the models do without essentially "reading" the label's definition
leak_cols = ['TMAX', 'TMIN', 'TEMP2M']

X_train_ablated = X_train.drop(columns=leak_cols)
X_val_ablated = X_val.drop(columns=leak_cols)
X_test_ablated = X_test.drop(columns=leak_cols)

ablated_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVC': SVC(probability=True, random_state=42)
}

ablated_trained = {}
for name, model in ablated_models.items():
    print(f"Training {name} (ablated)...")
    model.fit(X_train_ablated, y_train)
    ablated_trained[name] = model

ablated_val_results = {}
ablated_test_results = {}
for name, model in ablated_trained.items():
    ablated_val_results[name] = evaluate_model(model, X_val_ablated, y_val)
    ablated_test_results[name] = evaluate_model(model, X_test_ablated, y_test)

ablated_val_table = pd.DataFrame(ablated_val_results).T
ablated_test_table = pd.DataFrame(ablated_test_results).T

print("ABLATED VALIDATION COMPARISON (no TMAX/TMIN/TEMP2M):")
print(ablated_val_table)
print("\nABLATED TEST COMPARISON (no TMAX/TMIN/TEMP2M):")
print(ablated_test_table)

Training Logistic Regression (ablated)...
Training Decision Tree (ablated)...
Training Random Forest (ablated)...
Training Gradient Boosting (ablated)...
Training KNN (ablated)...
Training SVC (ablated)...
ABLATED VALIDATION COMPARISON (no TMAX/TMIN/TEMP2M):
                     Accuracy  Precision    Recall  F1-Score   ROC-AUC
Logistic Regression  0.929872   0.408333  0.890909  0.560000  0.976685
Decision Tree        0.944444   0.457944  0.593939  0.517150  0.778433
Random Forest        0.965392   0.630769  0.745455  0.683333  0.977142
Gradient Boosting    0.938676   0.442006  0.854545  0.582645  0.975800
KNN                  0.931087   0.408824  0.842424  0.550495  0.941518
SVC                  0.953552   0.520979  0.903030  0.660754  0.984799

ABLATED TEST COMPARISON (no TMAX/TMIN/TEMP2M):
                     Accuracy  Precision    Recall  F1-Score   ROC-AUC
Logistic Regression  0.935641   0.432961  0.945122  0.593870  0.976753
Decision Tree        0.950820   0.504545  0.676829  0.

This cell rebuilds the Voting and Bayesian ensembles using the top 3 ablated (leak-free) models based on validation F1-score, then combines their results with the individual model scores into final validation and test comparison tables. This gives the honest, final set of results to report, since it reflects performance without the temperature features that were inflating the earlier scores.

AI Disclaimer: AI (Gemini) was used to help write this code cell.
Prompt: "Write code that selects the top 3 models from an ablated validation results table by F1-Score, builds a soft VotingClassifier from them, fits it on the ablated training data, evaluates it on validation and test, then builds a Bayesian weighted ensemble using validation F1-scores as weights, evaluates that too, and finally combines all individual model results with both ensemble results into two final pandas DataFrames for validation and test."

In [10]:
# Rebuild ensembles on the ablated (leak-free) models
ablated_val_table_check = ablated_val_table.sort_values('F1-Score', ascending=False)
top_3_ablated = ablated_val_table_check.head(3).index.tolist()
print("Top 3 ablated models:", top_3_ablated)

top_3_ablated_estimators = [(name, ablated_trained[name]) for name in top_3_ablated]

voting_clf_ablated = VotingClassifier(estimators=top_3_ablated_estimators, voting='soft')
voting_clf_ablated.fit(X_train_ablated, y_train)

voting_val_ablated = evaluate_model(voting_clf_ablated, X_val_ablated, y_val)
voting_test_ablated = evaluate_model(voting_clf_ablated, X_test_ablated, y_test)

# Bayesian weighted ensemble
val_f1_ablated = np.array([ablated_val_table.loc[name, 'F1-Score'] for name in top_3_ablated])
weights_ablated = val_f1_ablated / val_f1_ablated.sum()
print("Ablated model weights:", dict(zip(top_3_ablated, weights_ablated)))

def bayesian_ensemble_predict_ablated(X):
    weighted_probs = np.zeros(len(X))
    for name, weight in zip(top_3_ablated, weights_ablated):
        probs = ablated_trained[name].predict_proba(X)[:, 1]
        weighted_probs += weight * probs
    return weighted_probs

val_probs_ablated = bayesian_ensemble_predict_ablated(X_val_ablated)
test_probs_ablated = bayesian_ensemble_predict_ablated(X_test_ablated)
val_preds_ablated = (val_probs_ablated >= 0.5).astype(int)
test_preds_ablated = (test_probs_ablated >= 0.5).astype(int)

bayesian_val_ablated = {
    'Accuracy': accuracy_score(y_val, val_preds_ablated),
    'Precision': precision_score(y_val, val_preds_ablated),
    'Recall': recall_score(y_val, val_preds_ablated),
    'F1-Score': f1_score(y_val, val_preds_ablated),
    'ROC-AUC': roc_auc_score(y_val, val_probs_ablated)
}
bayesian_test_ablated = {
    'Accuracy': accuracy_score(y_test, test_preds_ablated),
    'Precision': precision_score(y_test, test_preds_ablated),
    'Recall': recall_score(y_test, test_preds_ablated),
    'F1-Score': f1_score(y_test, test_preds_ablated),
    'ROC-AUC': roc_auc_score(y_test, test_probs_ablated)
}

print("Ablated Voting Ensemble - Val:", voting_val_ablated)
print("Ablated Voting Ensemble - Test:", voting_test_ablated)
print("Ablated Bayesian Ensemble - Val:", bayesian_val_ablated)
print("Ablated Bayesian Ensemble - Test:", bayesian_test_ablated)

# Final combined table
final_val_ablated = ablated_val_results.copy()
final_val_ablated['Voting Ensemble'] = voting_val_ablated
final_val_ablated['Bayesian Ensemble'] = bayesian_val_ablated

final_test_ablated = ablated_test_results.copy()
final_test_ablated['Voting Ensemble'] = voting_test_ablated
final_test_ablated['Bayesian Ensemble'] = bayesian_test_ablated

final_val_ablated_table = pd.DataFrame(final_val_ablated).T
final_test_ablated_table = pd.DataFrame(final_test_ablated).T

print("\nFINAL ABLATED VALIDATION TABLE:")
print(final_val_ablated_table)
print("\nFINAL ABLATED TEST TABLE:")
print(final_test_ablated_table)

# Save the final, leak-free comparison tables (these are the results that matter)
final_val_ablated_table.to_csv('validation_comparison.csv', index=True)
final_test_ablated_table.to_csv('test_comparison.csv', index=True)
print("\nFinal ablated comparison tables saved.")

Top 3 ablated models: ['Random Forest', 'SVC', 'Gradient Boosting']
Ablated model weights: {'Random Forest': np.float64(0.35465928290826154), 'SVC': np.float64(0.3429402400302697), 'Gradient Boosting': np.float64(0.3024004770614687)}
Ablated Voting Ensemble - Val: {'Accuracy': 0.9626593806921676, 'Precision': 0.59375, 'Recall': 0.806060606060606, 'F1-Score': 0.6838046272493573, 'ROC-AUC': np.float64(0.9823488964428562)}
Ablated Voting Ensemble - Test: {'Accuracy': 0.9614450516089861, 'Precision': 0.5731225296442688, 'Recall': 0.8841463414634146, 'F1-Score': 0.6954436450839329, 'ROC-AUC': np.float64(0.9848729837138628)}
Ablated Bayesian Ensemble - Val: {'Accuracy': 0.9626593806921676, 'Precision': 0.59375, 'Recall': 0.806060606060606, 'F1-Score': 0.6838046272493573, 'ROC-AUC': np.float64(0.9823915085660052)}
Ablated Bayesian Ensemble - Test: {'Accuracy': 0.9614450516089861, 'Precision': 0.5737051792828686, 'Recall': 0.8780487804878049, 'F1-Score': 0.6939759036144578, 'ROC-AUC': np.float

Summary and Discussion

This notebook trained six classification models plus two ensembles to predict HEATWAVE occurrence, and uncovered a data leakage issue along the way. With all features included, Decision Tree, Random Forest, Gradient Boosting, and both ensembles hit a suspicious 100% across every metric — investigation showed this was because TMAX, TMIN, and TEMP2M essentially encode the label's own definition, not genuine predictive signal.

After removing those three columns, Random Forest emerged as the strongest standalone model (F1 = 0.696, precision = 0.632 on test), while the Voting and Bayesian ensembles traded some precision for higher recall and the best ROC-AUC (0.985), making them better suited to a use case where missing a real heatwave is costlier than a false alarm. Precision remained the weak point across all models (0.51–0.63), meaning indirect atmospheric signals alone (wind, pressure, humidity, soil conditions) can flag likely heatwaves reasonably well but still generate a meaningful number of false positives.

The choice between Random Forest and the ensembles ultimately depends on the cost tradeoff between missed heatwaves and false alarms in a real early-warning use case — a point discussed further in the accompanying analysis report.